# Study A: calibration under different sampling designs

Study A tests a zero second information gap on the circle and sphere. The fitted-gap statistic is compared with score and Wald statistics, ordinary chi-squared calibration, and scalar effective-sample-size calibration.

A zero population gap means that the reduced model matches the added moments. It does not require that model to equal the full generating distribution. In particular, the fourth-harmonic example has higher-order structure despite a zero second gap.

Smoke mode uses five replicates in each of thirteen cells, for sixty-five replicates. Paper mode uses the full eleven thousand two hundred replicates. Both retain the original observations per replicate and the same fitting and covariance calculations.

The notebook calls the shared workflow and reads its saved results. It does not implement a second fitting routine. Generated outputs go to `results/smoke` or `results/paper`; the frozen `reference_results` directory is not overwritten.

Run all cells from top to bottom after changing the mode. Smoke mode checks execution and output structure. Its tiny simulation samples are not evidence for the manuscript's statistical conclusions.


In [1]:
# Change MODE to "paper" to run the complete manuscript experiment.
# Publication figures require LaTeX and the configured image-conversion tools.
MODE = "smoke"
FIGURES = False
assert MODE in {"smoke", "paper"}


In [2]:
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

cwd = Path.cwd().resolve()
REPO = next(
    (p for p in [cwd, *cwd.parents]
     if (p / "workflow.py").is_file() and (p / "code" / "gid_pipeline.py").is_file()),
    None,
)
if REPO is None:
    raise FileNotFoundError("Open this notebook from the repository root or its notebooks directory.")
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
from workflow import run_study


def read_json(path):
    return json.loads(Path(path).read_text())


def require_finite(frame, columns):
    values = frame.loc[:, columns].apply(pd.to_numeric, errors="raise").to_numpy()
    assert np.isfinite(values).all(), f"Nonfinite values in {columns}"


def show_run_figure(relative_path):
    if not FIGURES:
        print("Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.")
        return
    figure = RUN / relative_path
    if not figure.is_file():
        raise FileNotFoundError(f"The requested current-run figure was not generated: {figure}")
    display(Markdown(f"Figure from `{figure.relative_to(REPO)}` ({MODE} mode)."))
    display(Image(filename=str(figure)))

print(f"Repository root located: {REPO.name}")
print(f"Mode: {MODE}; figures: {FIGURES}")


Repository root located: github
Mode: smoke; figures: False


In [3]:
RUN = Path(run_study("a", mode=MODE, figures=FIGURES)).resolve()
assert RUN == (REPO / "results" / MODE).resolve()
assert RUN != (REPO / "reference_results").resolve()
print(f"Reading generated results from {RUN.relative_to(REPO)}")


smoke: study_ab-a


Reading generated results from results/smoke


## Verify the generated runs

The saved rows include fit validity, refinement status, and the design-specific covariance calculation. The checks below inspect those rows before displaying rejection frequencies.


In [4]:
summary = pd.read_csv(RUN / "study_a" / "rejection_summary.csv")
manifest = read_json(RUN / "study_a" / "numerical_manifest.json")
replicate_files = sorted((RUN / "study_a").glob("*_replicates.csv"))
replicates = pd.concat([pd.read_csv(p) for p in replicate_files], ignore_index=True)
expected = 65 if MODE == "smoke" else 11200
assert len(replicate_files) == 13
assert summary["cell"].nunique() == 13
assert len(replicates) == expected == manifest["total_replicates"]
assert not replicates.duplicated(["cell", "replicate"]).any()

display(replicates.groupby(["status", "refinement_status"], dropna=False).size().rename("replicates").to_frame())
assert replicates["valid"].astype(str).str.lower().eq("true").all(), "Inspect the saved invalid fits."
require_finite(replicates, ["gap", "p_gap", "p_score", "p_wald", "moment_error", "condition"])
assert replicates[["p_gap", "p_score", "p_wald"]].ge(0).all().all()
assert replicates[["p_gap", "p_score", "p_wald"]].le(1).all().all()
assert summary.drop_duplicates("cell")["repeats"].sum() == expected
print(f"Verified {expected} replicate rows across thirteen cells.")


,,replicates
status,refinement_status,
ok,ok,65


Verified 65 replicate rows across thirteen cells.


## Compare reference laws

Each rate uses the recorded simulation denominator. Wilson intervals describe Monte Carlo uncertainty in that rate. They are not confidence intervals for an individual information gap.

Informative weights can change covariance in ways that a scalar effective sample size cannot capture. Sampling-aware calibration therefore uses both fitted-model curvature and the observation design. Smoke-mode rates are too imprecise to assess calibration.


In [5]:
method_names = {
    "p_gap": "Fitted gap", "p_score": "Score", "p_wald": "Wald",
    "naive_p": "Ordinary chi-squared", "kish_p": "Scalar ESS",
}
rates = summary.pivot(index="cell", columns="method", values="rejection_rate")
display(rates.rename(columns=method_names))
display(summary.loc[summary["method"].eq("p_gap"),
    ["cell", "n", "repeats", "rejection_rate", "wilson_low", "wilson_high", "failures"]])


method,analytic_p,Scalar ESS,Ordinary chi-squared,Fitted gap,Score,Wald
cell,,,,,,
circle_cos4_direct_n200,0.0,0.0,0.0,0.0,0.0,0.0
circle_cos4_direct_n800,0.0,0.0,0.0,0.0,0.0,0.0
circle_uniform_direct_n200,NaN,0.2,0.2,0.2,0.2,0.2
circle_uniform_direct_n800,NaN,0.0,0.0,0.0,0.0,0.0
circle_uniform_importance_n200,NaN,0.0,0.4,0.0,0.0,0.0
circle_uniform_importance_n800,NaN,0.0,0.4,0.0,0.0,0.0
circle_vm2_direct_n200,NaN,0.0,0.0,0.0,0.0,0.0
circle_vm2_direct_n800,NaN,0.0,0.0,0.0,0.0,0.0
circle_vm2_importance_n200,NaN,0.0,0.0,0.0,0.0,0.0


,cell,n,repeats,rejection_rate,wilson_low,wilson_high,failures
0,circle_uniform_direct_n200,200,5,0.2,0.036224,0.624465,0
5,circle_vm2_direct_n200,200,5,0.0,0.000000,0.434482,0
10,circle_cos4_direct_n200,200,5,0.0,0.000000,0.434482,0
16,circle_uniform_importance_n200,200,5,0.0,0.000000,0.434482,0
21,circle_vm2_importance_n200,200,5,0.0,0.000000,0.434482,0
26,circle_uniform_direct_n800,800,5,0.0,0.000000,0.434482,0
31,circle_vm2_direct_n800,800,5,0.0,0.000000,0.434482,0
36,circle_cos4_direct_n800,800,5,0.0,0.000000,0.434482,0
42,circle_uniform_importance_n800,800,5,0.0,0.000000,0.434482,0
47,circle_vm2_importance_n800,800,5,0.0,0.000000,0.434482,0


## Numerical diagnostics

Reported maxima describe the checked runs. Unchecked refinements remain explicitly identified; these diagnostics do not certify an error bound over every possible sample.


In [6]:
diagnostic_keys = ["max_moment_error", "max_condition", "max_abs_kl_identity",
                   "max_scaled_refinement", "max_scaled_rotation", "max_reference_refinement"]
display(pd.Series({k: manifest[k] for k in diagnostic_keys}, name="observed maximum").to_frame())
surrogates = pd.read_csv(RUN / "study_a" / "surrogate_discrepancy.csv")
assert len(surrogates) == 13
display(surrogates)
print(manifest["disclaimer"])
show_run_figure("study_a/calibration_cdfs.png")


,observed maximum
max_moment_error,1.596729e-10
max_condition,1.440092e+03
max_abs_kl_identity,5.584766e-11
max_scaled_refinement,1.065814e-12
max_scaled_rotation,1.065814e-12
max_reference_refinement,1.385233e-03


,cell,mean,median,q05,q95,max_abs,median_ess,max_condition
0,circle_cos4_direct_n200,0.000795,0.000704,2.445152e-04,0.001435,0.001522,200.000000,1.245600
1,circle_cos4_direct_n800,0.000274,0.000003,8.663784e-07,0.000911,0.001063,800.000000,1.129870
2,circle_uniform_direct_n200,0.003437,0.000092,9.584332e-06,0.012593,0.015299,200.000000,1.483507
3,circle_uniform_direct_n800,0.000238,0.000160,1.573311e-05,0.000532,0.000570,800.000000,1.155734
4,circle_uniform_importance_n200,0.009358,0.003223,6.442762e-04,0.022595,0.024034,105.286392,1.416134
5,circle_uniform_importance_n800,0.001469,0.000177,2.685070e-05,0.003905,0.004161,414.365086,1.270470
6,circle_vm2_direct_n200,0.013303,-0.000038,-5.699688e-03,0.041151,0.043368,200.000000,14.381541
7,circle_vm2_direct_n800,0.010463,0.004491,-5.182737e-03,0.034191,0.039926,800.000000,18.643578
8,circle_vm2_importance_n200,0.000216,0.000281,-3.361995e-02,0.049462,0.060847,150.094521,11.850993
9,circle_vm2_importance_n800,0.008884,0.001694,-2.120933e-03,0.022707,0.022782,610.758619,12.402678


Observed refinement and rotation differences are diagnostics, not certified bounds on continuous-integration error.
Figure generation is off. Set FIGURES=True and rerun to create a figure from this run.
